In [1]:
from src.gillespie.simulation_config import SimulationConfig

/home/vmanso/PHD/moran/src/gillespie/simulation_config.py:108: SyntaxWarning: "is not" with 'float' literal. Did you mean "!="?
  scaled_omega_eshaust = config.omega_exhaust / self.OMEGA if config.omega_exhaust is not 0.0 else config.omega_exhaust


In [2]:
config= SimulationConfig()


In [3]:
print(config.cell_parameters["exhausted"].default_id)

(-2,)


In [4]:
print(config.cell_parameters["base"].K)

4000


In [5]:
from src.gillespie.clone_factory import CloneFactory
factory = CloneFactory(config= config)
test_clone = factory.create_clone(clone_id=(0,),clone_type= "mutated")
test_clone2 = factory.create_clone(clone_id=(1,), clone_type= "mutated", N=4)
test_clone3 = factory.create_clone(clone_id=(3,), clone_type= "mutated", N= None)

In [6]:
print(test_clone.N)
print(test_clone2.N)
print(test_clone3.N)

10
4
10


In [7]:
test4 = factory.create_clone(clone_type="mutated")
print(test4.cell_parameters)

CellTypeConfig(default_id=(-3,), N=10, K=8000, lambda0=0.005, mu=0.002, nu=0.0, omega_exhaust=0.0, next_mutation='')


In [8]:
from src.gillespie.tumor_simulation import TumorSimulation



In [9]:
simulation = TumorSimulation(config=config)
for cloneid, clones in simulation.tissue_state.clones.items():  
    print(cloneid, clones.clone_id)

arrancando con parametros:
SimulationConfig(OMEGA=4000, cell_parameters={'base': CellTypeConfig(default_id=(), N=0, K=4000, lambda0=0.005, mu=0.002, nu=0.0, omega_exhaust=0.0, next_mutation='mutated'), 'immune': CellTypeConfig(default_id=(-1,), N=0, K=2000, lambda0=0.005, mu=0.0, nu=0.0, omega_exhaust=7.5e-07, next_mutation=''), 'mutated': CellTypeConfig(default_id=(-3,), N=10, K=8000, lambda0=0.005, mu=0.002, nu=0.0, omega_exhaust=0.0, next_mutation=''), 'exhausted': CellTypeConfig(default_id=(-2,), N=0, K=None, lambda0=0.0, mu=0.002, nu=0.0, omega_exhaust=0.0, next_mutation='')}, T_max=10000, seed=None, decline=0.0, Kmin=1, fitness_gain=0.2, theta_I=1.25e-07, beta=1.0000000000000001e-07, d1_0=0.0, d2_0=0.0, instability_0=0.0, buildup_0=0.0, base_instability_buildup=0.0, mutation_instability_jump=0.0, mutation_buildup_gain=0.0, verbose=True, scale=True, decay=False, use_logistic=True, use_logistic_adapted=True, crowding_strategy=<src.gillespie.crowding_strategy.AdaptedCrowding object 

In [10]:
import pandas as pd

simulation.tissue_state.snapshot()

{(): {'Type': 'base',
  'N': 0,
  't': 0.0,
  'rb': 0.005,
  'rd': 0.002,
  'rm': 0.0,
  're': 0.0,
  'instability': 0.0,
  'buildup': 0.0},
 (-3,): {'Type': 'mutated',
  'N': 10,
  't': 0.0,
  'rb': 0.006,
  'rd': 0.002,
  'rm': 0.0,
  're': 0.0,
  'instability': 0.0,
  'buildup': 0.0},
 (-1,): {'Type': 'immune',
  'N': 0,
  't': 0.0,
  'rb': 0.005,
  'rd': 0.0,
  'rm': 0.0,
  're': 7.5e-07,
  'instability': 0.0,
  'buildup': 0.0},
 (-2,): {'Type': 'exhausted',
  'N': 0,
  't': 0.0,
  'rb': 0.0,
  'rd': 0.002,
  'rm': 0.0,
  're': 0.0,
  'instability': 0.0,
  'buildup': 0.0}}

In [13]:

# simulation.step()

# simulation.run()
# simulation.step()
# simulation.run()

for i in range(2):
    try:
        # Intentamos recuperar el clon y meter la mutación
        target_clone = simulation.tissue_state.clones[()]
        simulation._introduce_mutation(target_clone)
        print(f"Mutación introducida con éxito en la iteración {i}")
        
    except AssertionError as e:
        # Si el assert de tumor_simulation.py falla, atrapamos el error aquí
        print(f"Iteración {i} saltada: {e}")
        continue

Iteración 0 saltada: Cannot mutate a dead clone.
Iteración 1 saltada: Cannot mutate a dead clone.


In [14]:
beta = config.beta
mut=  simulation.tissue_state.pop_map.get("mutated")
imm = simulation.tissue_state.pop_map.get("immune")
print(mut)


10


In [15]:

print ( beta *mut*imm)

0.0


In [16]:

obj_imm = simulation.tissue_state.clones[(-3,)]
base = obj_imm.birth_rate * simulation.tissue_state.pop_map.get(obj_imm.get_type(),0) 
crowding_effect = obj_imm.config.crowding_strategy.crowding(obj_imm,tissue_state=simulation.tissue_state)
print(obj_imm.crowding_numerator(simulation.tissue_state))#


10


In [17]:

print(obj_imm.actual_K)


13334


In [ ]:

print(base)


0.084


In [ ]:

print(crowding_effect)


0.9989500524973751


In [ ]:
#TODO: Improve efficiency: right now every clone has its own id, which is fine but we need to store the calculation of the rates inside the subclass instead of running it for each clone. (calculate once the rates at each time and pass it to all subclasses )
simulation.tissue_state.print_pop_map()
for clones in simulation.tissue_state.clones.values():
    print("--------------------------------")
    print(clones)
    print(clones.clone_id)
    print(clones.cell_parameters)
    print(clones.actual_K)
    print(clones.birth_rate_effective(simulation.tissue_state))
    

clone_type | count
----------+------
base      | 8
exhausted | 0
immune    | 0
mutated   | 14
--------------------------------
base
()
CellTypeConfig(default_id=(), N=10, K=4000, lambda0=0.005, mu=0.002, nu=0.0, omega_exhaust=0.0, next_mutation='mutated')
6667
0.03986800659967002
--------------------------------
mutated
(-3,)
CellTypeConfig(default_id=(-3,), N=10, K=8000, lambda0=0.005, mu=0.002, nu=0.0, omega_exhaust=0.0, next_mutation='')
13334
0.08391180440977951
--------------------------------
immune
(-1,)
CellTypeConfig(default_id=(-1,), N=0, K=2000, lambda0=0.005, mu=0.0, nu=0.0, omega_exhaust=7.5e-07, next_mutation='')
2000
0.0
--------------------------------
exhausted
(-2,)
CellTypeConfig(default_id=(-2,), N=0, K=None, lambda0=0.0, mu=0.002, nu=0.0, omega_exhaust=0.0, next_mutation='')
inf
0.0
--------------------------------
mutated
(1,)
CellTypeConfig(default_id=(-3,), N=10, K=8000, lambda0=0.005, mu=0.002, nu=0.0, omega_exhaust=0.0, next_mutation='')
13334
0.08391180440977

In [ ]:
simulation.tissue_state.clones[(-3,)].crowding_numerator(simulation.tissue_state)

14

In [ ]:
simulation.tissue_state.clones[(-3,)].N

10